# Modellierungsseminar Sommer 2026
## Cycle Planning for workforce scheduling

In [1]:
# install Gurobi package in case not done yet:
%pip install gurobipy 
import gurobipy as gp
from gurobipy import GRB
import pandas as pd
from dataclasses import dataclass
import datetime as dt
from pathlib import Path # for easier and robust folder and file handling across OS (Path can be used by Pandas directly)
import src.Shift as Shift # tailor-made data type for shift definitions
import src.functions as abd # self-made functions by Arty, Ben and Dirk... ;-) => call them by starting with "abd."


Note: you may need to restart the kernel to use updated packages.


### Read parameters

In [2]:
# determine folder structure for inputs and outputs
PROJECT_ROOT = Path.cwd() # main folder of the code
FOLDER_INPUT =  PROJECT_ROOT / "input" # data input
FOLDER_LOGS = PROJECT_ROOT / "logs" # folder for log files, eg exorts of data sets for more transparency
FOLDER_OUTPUT = PROJECT_ROOT / "output/"
FOLDER_AND_FILE_LOG =  PROJECT_ROOT / "logs" / "cyclePlanning_logs.txt"
abd.writeToLogs("cycle planning process started", FOLDER_AND_FILE_LOG, deleteHistory=True) # very first log entry deletes old log entries

In [3]:
# load parameters from CSV into dict
params = abd.readParameters(FOLDER_INPUT / "parameters.csv")
abd.writeToLogs("parameters loaded", FOLDER_AND_FILE_LOG)

### Global variables

In [4]:
# global static variables

# more relevant for more complex models going forward
MAX_CYCLE_WEEKS = int(params["max_cycle_length"])   # max number of cycle weeks
MIN_REST        = int(params["min_rest"])            # min rest time between shifts in minutes
MAX_CONSEC_DAYS = int(params["max_conseq_working_days"])  # max consecutive working days

# variables for hourly fairness and average weekly work hours
AVG_WEEKLY_HOURS    = float(params["avg_weekly_work_hours"])   # target avg weekly work hours
AVG_REFERENCE_WEEKS = int(params["avg_reference_weeks"])       # weeks window for average (currently unused, reserved)
#MAX_WEEKLY_HOURS    = AVG_WEEKLY_HOURS * AVG_REFERENCE_WEEKS   # OLD: 40*4 = 160h against a SINGLE week
# FIX (dead constraint c06): one week holds at most 7 shifts (c02), far below 160h,
# so the old cap could never bind and c06 silently did nothing. The cap is now a
# real user parameter (team decision: hard 56h limit per cycle week).
MAX_WEEKLY_HOURS    = float(params["max_weekly_work_hours"])   # hard cap on work hours per cycle week
MAX_FREE_DAYS_PER_WEEK = int(params["max_free_days_per_week"]) # NEW: max fully free days per active week (team: 2)
W_NB_WORKERS  = int(params["obj_w_workers"])   # weight: minimize active cycle weeks (across all cycles)
W_PESCH1 = int(params["obj_w_pesch1"])  # weight: Pesch1 deviation from Soll weekly hours
W_PESCH2 = int(params["obj_w_pesch2"])  # weight: Pesch2 equal shift-class proportion across cycles
W_CHANGEofCLASSES = int(params["obj_w_minChangeOfClasses"]) # weight: objective to minimize change of shift classes 
W_PESCH3 = int(params["obj_w_pesch3"])  # weight: Pesch3 minimize day-by-day changes of the shift TYPE

#MAX_CYCLE_WEEKS = 365  # in future this shall be user's input => too long snakes do not help to ensure "fair" distribution of shifts
MAX_NB_CYLCEs = int(params["max_nb_cycles"]) # number of cycles

DICT_WEEKDAYS = {'Mon':1,'Tue':2,'Wed':3,'Thu':4,'Fri':5,'Sat':6,'Sun':7,
                 'Monday':1,'Tuesday':2,'Wednesday':3,'Thursday':4,'Friday':5,'Saturday':6,'Sunday':7,
                 'Mo':1,'Tu':2,'We':3,'Th':4,'Fr':5,'Sa':6,'Su':7,
                 '1':1,'2':2,'3':3,'4':4,'5':5,'6':6,'7':7
                 }
DICT_WEEKDAYS_RETURN = {1: "Mon", 2: "Tue", 3: "Wed", 4: "Thu", 5: "Fri", 6: "Sat", 7: "Sun"}



In [5]:
# basic inputs and parameters
cycles = range(1, MAX_NB_CYLCEs+1)
cycleWeeks  = range(1, MAX_CYCLE_WEEKS+1)
Weekdays = range(1,8)  # results in 1,...,7 => let 1 be Monday and 7 be Sunday (in line with static variable DICT_WEEKDAYS)

### Shift Objects
creating shift objects:

In [6]:

# read input data for shift definitions
data_shiftSet = abd.readShiftSet(FOLDER_INPUT / "input_ShiftDataSet_Pesch.csv")
shift_objects = abd.build_shift_objects(data_shiftSet)

# optional enhancement: 
# use user-input for number of stand-bys 
# OR deviate need from the number of shifts to be staffed on a specific day (eg 5%)
reserveShift = Shift.Shift("[reserveShift]", 
                           "dummy for jump shifts", 
                           ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], 
                           dt.time(*map(int,"07:00".split(':'))),
                           dt.time(*map(int,"07:00".split(':'))),
                           1, 
                           #5,  # OLD: collided with [00dayweekend] (also class 5), so Pesch2/Pesch8 merged them
                           4,   # PROVISIONAL own class for reserve; team still owes the real Beliebtheit value
                           8, 
                           1, 
                           None 
                           )
shift_objects.append(reserveShift)

# distuingish work shift from all shifts: 
#   WorkShifts are all shifts imported from the shift set file plus reserveShifts
#   other shifts are hard-coded below (freeDay)
WorkShifts = [s.shift_id for s in shift_objects] #object oriented solution
abd.writeToLogs(f"WorkShifts are defined as {WorkShifts}", FOLDER_AND_FILE_LOG)


# additional hard-coded shifts for stand-bys and free days
freeDayShift = Shift.Shift("[freeDay_:-)_]", 
                           "dummy shift for free days", 
                           ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"], 
                           dt.time(*map(int,"07:00".split(':'))),
                           dt.time(*map(int,"07:00".split(':'))),
                           0, 
                           1, 
                           0, 
                           0, 
                           None 
                           )
shift_objects.append(freeDayShift)


#log
abd.writeDataToLogs(shift_objects, FOLDER_LOGS / "log_shift_object.csv")

Shifts = [s.shift_id for s in shift_objects]
abd.writeToLogs(f"    Shifts are defined as {Shifts}", FOLDER_AND_FILE_LOG)

In [7]:
# preparation for output
shift_info = {
    s.shift_id: {
        "start": s.start.strftime("%H:%M"),
        "end": s.end.strftime("%H:%M"),
        "workingTime": s.shift_work_time_assignment
    }
    for s in shift_objects
}

In [8]:
# Pre-compute all shift pairs that violate MIN_REST if scheduled on consecutive days.
# Excludes dummy shifts (freeDay, spareShift) since they have no real start/end times.
# Result: list of (sh1_id, sh2_id) tuples that cannot appear on consecutive days in a snake.
DUMMY_SHIFTS = {"[freeDay_:-)_]", "[reserveShift]"}
incompatible_pairs = [
    (sh1.shift_id, sh2.shift_id)
    for sh1 in shift_objects if sh1.shift_id not in DUMMY_SHIFTS
    for sh2 in shift_objects if sh2.shift_id not in DUMMY_SHIFTS
    if Shift.rest_minutes_between(sh1, sh2) < MIN_REST
]

# Team decision (2026-07-07): reserveShift must NOT directly follow a night shift.
# reserve is a joker: the person can be pulled into ANY shift, including a night,
# and two nights in a row are illegal (min rest 11h vs the 9h15 gap between nights).
# reserve carries placeholder clock times, so the automatic rest computation above
# cannot see this case; the forbidden pairs are added explicitly. c05 (and the ring
# closure cell) then enforce them like any other incompatible pair.
overnight_ids = [s.shift_id for s in shift_objects
                 if s.shift_id not in DUMMY_SHIFTS and s.end <= s.start]
incompatible_pairs += [(night_id, reserveShift.shift_id) for night_id in overnight_ids]


In [9]:
# Pre-compute work hours per shift using shift_duration_hours from Shift.py.
#
# FIX (reserve hours, 2026-07-07): reserveShift is REAL work time (24h standby,
# the person must be ready to jump in at any moment), but it used to be excluded
# here together with freeDay via DUMMY_SHIFTS, so its hours were invisible to
# Pesch1, c06 and the output. Only the free day truly carries no hours.
# DUMMY_SHIFTS (previous cell) still excludes both from the automatic MIN_REST
# pairing, because their clock times are placeholders.
NO_HOURS_SHIFTS = {freeDayShift.shift_id}
shift_hours = {
    s.shift_id: Shift.shift_duration_hours(s)
    for s in shift_objects
    #if s.shift_id not in DUMMY_SHIFTS   # OLD: also dropped reserveShift -> its 24h were lost
    if s.shift_id not in NO_HOURS_SHIFTS # FIX: only the free day has no hours
}

# net work time per shift (break excluded); fall back to presence time if not specified ('none')
work_hours = {
    s.shift_id: (float(s.shift_work_time_assignment)
                 if s.shift_work_time_assignment is not None
                 else Shift.shift_duration_hours(s))
    for s in shift_objects
    #if s.shift_id not in DUMMY_SHIFTS   # OLD: also dropped reserveShift -> its 24h were lost
    if s.shift_id not in NO_HOURS_SHIFTS # FIX: only the free day has no hours
}

#when none then gleich wie shift hours 

### modelling

initiate model and basic decision variables

$x \to$ x  
$y \to$ active\_week  
$z \to$ active\_cycle  

In [10]:
# modelling

modelCycle = gp.Model("SnakeBuilding_simple")

# --- decision variables
# x[c, s, d, sh] = 1 if in cycle c, cycleWeek s, on weekday d, shift sh is assigned
x = modelCycle.addVars(cycles, cycleWeeks, Weekdays, Shifts,
                       vtype=GRB.BINARY, name="x")

# active_cycle[c] = 1 if cycle c is used at all, 0 otherwise
active_cycle = modelCycle.addVars(cycles, vtype=GRB.BINARY, name="active_cycle")

# active_week[c,s] = 1 if cycle c uses cycleWeek s (i.e., at least one shift in that week is active), 0 otherwise
active_week = modelCycle.addVars(cycles, cycleWeeks, vtype=GRB.BINARY, name="active_week")


Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2810721
Academic license 2810721 - for non-commercial use only - registered to ar___@student.uni-siegen.de


### constraints:

#### basic constraints

$ y_{c,w} - z_w <= 0$,  $ \forall c $  $ \forall w $  

In [11]:
# basic constraints

# ensure a cycleWeek can only be active when the according cylce is active
for c in cycles:
    for w in cycleWeeks:
        modelCycle.addConstr(active_week[c,w] - active_cycle[c] <= 0,
                             name=f"WeekImpliesCycle_c{c}_s{w}")

$ \sum_{d \in D} \sum_{s \in S}{x_{c,w,d,s}} \leq |D| \cdot |S| \cdot y_{c,w}  $  ,  $ \forall c $  $ \forall w $  

In [12]:
# enforce active_week >= any assignment in that week
for c in cycles:
    for w in cycleWeeks:
        modelCycle.addConstr(
            gp.quicksum(x[c, w, d, sh] for d in Weekdays for sh in Shifts) <= len(Weekdays) * len(Shifts) * active_week[c, w],
            name=f"Link_x_activeWeek_c{c}_s{w}_upper"
        )

In [13]:
# cycle must be active if any week in that cycle is active
# "no active cycleWeek without active cycle"
for c in cycles:
    modelCycle.addConstr(
        gp.quicksum(active_week[c, s] for s in cycleWeeks) <= MAX_CYCLE_WEEKS * active_cycle[c],
        name=f"Link_activeWeek_activeCycle_upper_c{c}"
    )
    modelCycle.addConstr(
        gp.quicksum(active_week[c, s] for s in cycleWeeks) >= active_cycle[c],
        name=f"Link_activeWeek_activeCycle_lower_c{c}")


ensure that cycles and cycleWeeks are activated in ascending order  

$y_s - y_{s+1} >= 0, \forall s \in \{1,\dots,M-1\}, \forall c \in C$  
$z_c - z_{c+1} >= 0, \forall c \in \{1,\dots,C-1\}$

In [14]:

for c in range(1, MAX_NB_CYLCEs): # loop from first to second-last entry
    modelCycle.addConstr(active_cycle[c] - active_cycle[c + 1] >= 0,
                         name=f"CycleOrder_c{c}")

for c in cycles: # loop across all possible cycles
    for w in range(1, MAX_CYCLE_WEEKS): # loop from first to second-last entry
        modelCycle.addConstr(active_week[c, w] - active_week[c, w + 1] >= 0,
                             name=f"WeekOrder_c{c}_s{w}")


#### condition c01

each shift has to be covered on each day exactly once

$\sum_{s=1}^{n}{x_{c,w,d,s}} >= 1$  
$ \forall d \in D,$  
$ \forall w \in N,$  
$ \forall c \in M$  

$x_{c,w,d,s} = 1$, when cycle c is working in cycleWeek w on day d, covering shift s

$x: $ binary variable,
$c: $ cycle,
$d: $ weekday,
$w: $ cycleWeek
$s: $ shift


In [15]:
for d in Weekdays: #loop over all week days
    for ws in WorkShifts: # loop over all work shift
        shift = next(s for s in shift_objects if s.shift_id == ws) # 'next()' is an alternative for 'for s in shift_objects: if s.shift_id == ws: shift = s'
#       if DICT_WEEKDAYS[shift.weekdays[0]] <= d <= DICT_WEEKDAYS[shift.weekdays[-1]]:
        if d in [DICT_WEEKDAYS[w] for w in shift.weekdays]: # correction to also cover shifts that appear for non-consecutive week days (eg. Mon, Wed)
            modelCycle.addConstr(gp.quicksum(x[c, s, d, ws] for c in cycles for s in cycleWeeks) >= 1,
                        name=f"Cover_day{d}_{ws}")
        else:
            modelCycle.addConstr(gp.quicksum(x[c, s, d, ws] for c in cycles for s in cycleWeeks) <= 0,
                        name=f"Cover_day{d}_{ws}")

#Логика: перед добавлением ограничения проверяем входит ли день d в список weekdays этой смены. Если нет ограничение не добавляется, смена в этот день не требуется.
#Logic: Before adding a restriction, we check whether day d is included in the list of weekdays for this shift. If not, the restriction is not added, as no shift is required on that day.

# 1. each shift has to be covered on each day
#for d in Weekdays:
#    for ws in WorkShifts:
#        m.addConstr(gp.quicksum(x[s, d, ws] for s in range(MAX_CYCLE_WEEKS)) >= 1,
#                    name=f"Cover_day{d}_{ws}")



#### condition c02

<div align="left">

$ \sum_{s \in S}{x_{c,w,d,s}} - y_s = 0$  
$ \forall c \in C, \forall w \in W, \forall d \in D $

</div>

In [16]:
# 2. each cycle shall have exactly one shift per day (implying that on cycleWeek has exactly one shift per day)
for c in cycles:
    for w in cycleWeeks:
        for d in Weekdays:
            modelCycle.addConstr(
                gp.quicksum(x[c, w, d, sh] for sh in Shifts) - active_week[c, w] == 0,
                name=f"OneShiftPerDay_c{c}_s{w}_d{d}"
            )


#### condition c03

$ \sum_{d = 1}^{8-MAX\_CONSEC\_DAYS+1}{x_{}} $  
$ \forall c \in C, \forall w \in W, \forall d \in {1,\dots, 8 - MAX\_CONSEC\_DAYS +1} $

In [17]:
### UPDATED // DOUBLE-CHECK!!!
### => is there a minimum rest time after 5 days working???
### => or should this condition rather be "max 5 working days within 7 days"???

# Helper function: convert a global day index t into tupel (cycleWeek, weekday)
# This allows to treat all days across all cycleWeeks as one continuous timeline.
def decode_global_day(t):
    s = (t - 1) // 7 + 1 # Compute cycleWeek index from global day t (1-based indexing)
    d = (t - 1) % 7 + 1 # Compute weekday index from global day t (1-based indexing)
    return s, d

# total number of days across all cycleWeeks
# used to define the global timeline over which consecutive work days are checked.
TOTAL_DAYS = MAX_CYCLE_WEEKS * 7

# limit the number of consecutive working days across all cycleWeeks
# for each cycle scan through the entire global timeline and check every (MAX_CONSEC_DAYS + 1)-days window
# to ensure that not all days in the window are working days.
for c in cycles:

    # iterate over all possible start positions of a sliding window
    # the window length is MAX_CONSEC_DAYS + 1, so the last valid start is:
    # TOTAL_DAYS - MAX_CONSEC_DAYS
    for t_start in range(1, TOTAL_DAYS - MAX_CONSEC_DAYS + 1):

        # Collect expressions representing work indicators for each day in the window
        moving_time_window = []

        # iterate through each offset inside the window
        # offset = 0 means the first day of the window
        # offset = MAX_CONSEC_DAYS means the last day of the window
        for offset in range(0, MAX_CONSEC_DAYS + 1):

            # Compute the global day index inside the window
            t = t_start + offset

            # Convert global day index back to (cycleWeek, weekday)
            s, d = decode_global_day(t)

            # work[c,s,d] = sum of all work shifts assigned on that day
            # If any WorkShift is assigned, this sum becomes 1 (binary model)
            moving_time_window.append(
                gp.quicksum(x[c, s, d, sh] for sh in WorkShifts)
            )

        # ACTUAL CONSTRAINT COMES HERE:
        # In any time window of length MAX_CONSEC_DAYS + 1,
        # the number of working days must be <= MAX_CONSEC_DAYS.
        # This prevents sequences of MAX_CONSEC_DAYS + 1 consecutive working days.
        modelCycle.addConstr(
            gp.quicksum(moving_time_window) <= MAX_CONSEC_DAYS,
            name=f"MaxConsecDays_c{c}_t{t_start}"
        )


### QUESTION: How many days rest after 5 days working?

#### Condition c05

In [18]:
# c05: minimum rest time between consecutive shifts within a snake week.
# If sh1 on day d and sh2 on day d+1 violate MIN_REST, they cannot both be assigned to the same snake.
# Covers days 1-6 only; wrap-around (day 7 -> day 1) not yet modelled. => DONE NOW
for c in cycles:
    for w in cycleWeeks:
        for d in Weekdays:
            if d <= 6:  # for Mon to Sat use pairs of day d and d+1
                for (sh1, sh2) in incompatible_pairs:
                    modelCycle.addConstr(
                        x[c, w, d, sh1] + x[c, w, d+1, sh2] <= 1,
                        name=f"MinRest_c{c}_w{w}_d{d}_{sh1}_{sh2}"
                    )
            elif d == 7 and w < MAX_CYCLE_WEEKS:  # Sun of week w -> Mon of NEXT week w+1 (snake runs continuously)
                for (sh1, sh2) in incompatible_pairs:
                    modelCycle.addConstr(
                        x[c, w, d, sh1] + x[c, w+1, 1, sh2] <= 1,
                        name=f"MinRest_c{c}_w{w}_d{d}_{sh1}_{sh2}"
                    )
            # FIX (C05 wrap-around): the Sun->Mon transition below originally paired
            # Sunday of week w with Monday of the SAME week w. In a snake the weeks run
            # continuously, so Sunday of week w is followed by Monday of the NEXT week (w+1).
            # The old version constrained a non-existent backward pair and left the real
            # week-to-week rest gap unchecked. Now paired with w+1, guarded by w < MAX_CYCLE_WEEKS
            # (the last week has no successor). Old code kept commented below for reference.          

#### Condition c06

In [19]:
# c06: total work hours per active cycle week must not exceed MAX_WEEKLY_HOURS.
# NOTE (2026-07-07): MAX_WEEKLY_HOURS is now the hard 56h user parameter (see the
# globals cell). The old value (40h * 4 weeks = 160h) was applied to a SINGLE week
# and could never be reached, so this constraint used to be permanently slack.
# ensures no cycle week accumulates more than the allowed weekly work time.
# bound scales with active[s] so inactive cycles are not constrained.
for c in cycles:
    for w in cycleWeeks:
        modelCycle.addConstr(
            gp.quicksum(
               #shift_hours.get(sh, 0) * x[c, w, d, sh] # replaced by more efficient version without "get()":
                work_hours[sh] * x[c, w, d, sh]
                for d in Weekdays
               #for sh in Shifts if sh in shift_hours # # replaced by more efficient version:
                for sh in work_hours.keys()
            ) <= MAX_WEEKLY_HOURS * active_week[c, w],
            name=f"MaxWeeklyHours_c{c}_w{w}"
        )

#### Condition c07  
ensure balanced [almost equal] lenght of cycles (same number of cycleWeeks plus/minus 1 week)

$ W_c = \sum_{w=1}^{M} y_{c,w} \text{  : counting active weeks } w \text{ within cycle } c$  

$M: \text{MAX\_CYCLE\_WEEKS}$  

$\text{condition: }|W_c - W_{c'}| \le 1 \quad \forall c,c' \in C$  

$\text{linearized condition: }$  
$W_c - W_{c'} \le 1 \quad \forall c,c' \in C$  
$W_{c'} - W_c \le 1 \quad \forall c,c' \in C$

In [20]:
# c07: ensure all used cycles have balanced number of cycleWeeks;
# deviations of at most 1 week are allowed

# determine number of active weeks per cycle
# (W_c in formula)
cycle_length = {c: gp.quicksum(active_week[c, s] for s in cycleWeeks) for c in cycles}

# pairwise balance constraints
# |W_c1 - W_c2| <= 1  for all cycles c1 != c2 => implemented as 2 linear statements
for c1 in cycles:
    for c2 in cycles:
        if c1 < c2:  # avoiding duplicates and self-pairing
            # W_c1 - W_c2 <= 1
            modelCycle.addConstr(
                cycle_length[c1] - cycle_length[c2] <= 1 + MAX_CYCLE_WEEKS * (1 - active_cycle[c2]), name=f"CycleBalance_upper_c{c1}_c{c2}") ## upper bound: cycle c1 cannot be more than 1 week longer than c2 — relaxed if c2 is inactive
            # W_c2 - W_c1 <= 1
            modelCycle.addConstr(cycle_length[c2] - cycle_length[c1] <= 1 + MAX_CYCLE_WEEKS * (1 - active_cycle[c1]), name=f"CycleBalance_lower_c{c1}_c{c2}") ## lower bound: cycle c2 cannot be more than 1 week longer than c1 — relaxed if c1 is inactive
# Big-M: if cycle c is inactive (active_cycle=0), the right side becomes 1+MAX_CYCLE_WEEKS, which is always larger than any possible difference — so the constraint has no effect

#### condition c08
max fully free days per active week (team decision 2026-07-07)

In [21]:
# c08: at most MAX_FREE_DAYS_PER_WEEK completely free days per active week.
# Team decision (2026-07-07): a snake week must stay a real working week; a worker
# gets at most 2 fully free days per week ("free" = not even standby duty; the
# reserveShift does NOT count here, it is work).
# Known and intended interaction: a night shift can only be followed by a free day
# (min rest kills night->day and night->night, and the team banned night->reserve),
# so this cap indirectly limits nights to about 3 per week and makes "pure night
# weeks" impossible. That is deliberate: the load should stay mixed and fair.
# For inactive weeks the right-hand side is 0 and c02 already forces all x to 0.
for c in cycles:
    for w in cycleWeeks:
        modelCycle.addConstr(
            gp.quicksum(x[c, w, d, freeDayShift.shift_id] for d in Weekdays)
            <= MAX_FREE_DAYS_PER_WEEK * active_week[c, w],
            name=f"MaxFreeDays_c{c}_w{w}"
        )

In [22]:
# Pesch1: deviation of average weekly net work time from target (AVG_WEEKLY_HOURS)
#cycle_hours = {
#    c: gp.quicksum(work_hours[sh] * x[c, w, d, sh]
#                   for w in cycleWeeks for d in Weekdays for sh in work_hours.keys())
#    for c in cycles
#}
#dev_pos = modelCycle.addVars(cycles, lb=0, name="dev_pos")
#dev_neg = modelCycle.addVars(cycles, lb=0, name="dev_neg")
#
#for c in cycles:
#    modelCycle.addConstr(
#        cycle_hours[c] - AVG_WEEKLY_HOURS * cycle_length[c] == dev_pos[c] - dev_neg[c],
#        name=f"Pesch1_dev_c{c}"
#    )

#####
# Pesch1: per-week deviation of net work time from the weekly target (AVG_WEEKLY_HOURS).
# Soft target: the objective penalises deviation, it does NOT force exactly 40h.
# Measured per week so heavy/light weeks cannot cancel out across the cycle.
week_hours = {
    (c, w): gp.quicksum(work_hours[sh] * x[c, w, d, sh]
                        for d in Weekdays for sh in work_hours.keys())
    for c in cycles for w in cycleWeeks
}

dev_pos = modelCycle.addVars(cycles, cycleWeeks, lb=0, name="dev_pos")
dev_neg = modelCycle.addVars(cycles, cycleWeeks, lb=0, name="dev_neg")

for c in cycles:
    for w in cycleWeeks:
        modelCycle.addConstr(
            week_hours[(c, w)] - AVG_WEEKLY_HOURS * active_week[c, w] == dev_pos[c, w] - dev_neg[c, w],
            name=f"Pesch1_dev_c{c}_w{w}"
        )



In [23]:
# Pesch2: equal proportion of shift categories (by presence time) across cycles.
# Category = shift_class value. Uses shift_hours (presence / Anwesenheitszeit), NOT work_hours.
#shift_class_of = {s.shift_id: s.shift_class for s in shift_objects if s.shift_id not in DUMMY_SHIFTS}  # OLD: dropped reserve
# FIX (2026-07-07): shift_hours now includes reserveShift (it is real work), so the
# class map must cover it too, otherwise the cat_hours lookup below crashes.
# reserve currently shares class 5 with the weekend day shift (open team question).
shift_class_of = {s.shift_id: s.shift_class for s in shift_objects if s.shift_id not in NO_HOURS_SHIFTS}
categories = sorted(set(shift_class_of.values()))

# presence hours of each category k inside each cycle c
cat_hours = {
    (c, k): gp.quicksum(shift_hours[sh] * x[c, w, d, sh]
                        for w in cycleWeeks for d in Weekdays
                        for sh in shift_hours.keys() if shift_class_of[sh] == k)
    for c in cycles for k in categories
}

# ---- Pesch2 v2 (2026-07-07): each cycle vs the GLOBAL class mix -------------
# The old pairwise version (hashed below) only balanced neighbouring cycles
# (1~2, 2~3), so the outer cycles could drift apart by twice the tolerance, and
# it needed a Big-M (grown to ~3000 after reserve got hours) that hurt numerics.
# The spec formula instead compares each cycle's class share with the share p_k
# of that class in the WHOLE dataset. p_k is a plain data constant, so
# p_k * total_pres stays linear, and an inactive cycle gives 0 == 0 by itself:
# no Big-M needed at all.

# p_k: share of class k in the dataset's coverage-required presence hours
dataset_class_hours = {k: 0.0 for k in categories}
for s in shift_objects:
    if s.shift_id in shift_class_of:
        dataset_class_hours[shift_class_of[s.shift_id]] += shift_hours[s.shift_id] * len(s.weekdays)
p = {k: dataset_class_hours[k] / sum(dataset_class_hours.values()) for k in categories}

# total presence hours of each cycle (sum of its category hours)
total_pres = {c: gp.quicksum(cat_hours[c, k] for k in categories) for c in cycles}

d2_pos = modelCycle.addVars(cycles, categories, lb=0, name="pesch2_pos")
d2_neg = modelCycle.addVars(cycles, categories, lb=0, name="pesch2_neg")

# deviation of class k in cycle c from its fair share of that cycle's hours
for c in cycles:
    for k in categories:
        modelCycle.addConstr(
            cat_hours[c, k] - p[k] * total_pres[c] == d2_pos[c, k] - d2_neg[c, k],
            name=f"Pesch2_share_c{c}_k{k}"
        )

# OLD pairwise version, kept for reference:
## Balance category hours between consecutive cycles (c04 chains them: c1 ~ c2 ~ c3).
## Soft: deviation goes into the objective. Big-M relaxes the pair when the higher
## cycle is inactive (same trick as c07), so we only balance actually-used cycles.
#pair_cycles = range(1, MAX_NB_CYLCEs)  # pairs (c, c+1)
#d2_pos = modelCycle.addVars(pair_cycles, categories, lb=0, name="pesch2_pos")
#d2_neg = modelCycle.addVars(pair_cycles, categories, lb=0, name="pesch2_neg")

#BIG_M_HOURS = MAX_CYCLE_WEEKS * 7 * max(shift_hours.values())

#for c in pair_cycles:
#    for k in categories:
#        diff = cat_hours[c, k] - cat_hours[c+1, k]
#        modelCycle.addConstr(diff - (d2_pos[c, k] - d2_neg[c, k]) <=  BIG_M_HOURS * (1 - active_cycle[c+1]),
#                             name=f"Pesch2_bal_up_c{c}_k{k}")
#        modelCycle.addConstr(diff - (d2_pos[c, k] - d2_neg[c, k]) >= -BIG_M_HOURS * (1 - active_cycle[c+1]),
#                             name=f"Pesch2_bal_lo_c{c}_k{k}")

#### Pesch8 / minimize day-to-day changes of shift class  

$$ \sum_{c=1}^{M}{x_{c,w,d,s}},\text{ } \forall w,d,s$$

In [24]:
# pre-work for condition to have as less shift class changes as possible (target: same shift-class in blocks as far as possible)

# Map each shift id to its shift_class (eg 'dayShift': 1)
dict_shift_to_class = {s.shift_id: s.shift_class for s in shift_objects}

# unique list of shift classes from shift definition (from input file)
set_classes = sorted(set(dict_shift_to_class.values()))


In [25]:
# additional decision variables to link shift classes to active shifts/shiftWeek/weekDay/cycle:

# binary variable indicating whether a specific class is assigned on (c,w,d), ie in cycle c, in cycleWeek w, on weekDay d
# class_assigned[c,w,d,k] = 1 if on cycle c, week w, weekday d the assigned shift belongs to class k
class_assigned = modelCycle.addVars(cycles, cycleWeeks, Weekdays, set_classes, vtype=GRB.BINARY, name="class_assigned")

# add supporting variables for absolute differences per class between consecutive days
# diff[c,w,d,k] >= | class_assigned[c,w,d,k] - class_assigned[c,w,d+1,k] |
class_diff = modelCycle.addVars(cycles, cycleWeeks, Weekdays, set_classes, vtype=GRB.BINARY, name="diff_class")


In [26]:
# link class_assigned to x
# for each shiftClass k, class_assigned equals the sum of x over all shifts that belong to that class
# this enforces that class_assigned is 1 exactly when a shift of that class is chosen on that day
dict_shifts_by_class = {k: [sh for sh, shiftClass in dict_shift_to_class.items() if shiftClass == k and sh in WorkShifts] for k in set_classes}
# results in sth like this:
    # 2: ["[00day000week]%_%1", "[00day000week]%_%2", ...],
    # 5: ["[00dayweekend]%_%1", ...],
    # 7: ["[night000week]%_%1", "[night000week]%_%2", ...],
    # 9: ["[nightweekend]%_%1", ...],
    #10: ["[allDay_shift]%_%1"]
    # => focus on WorkShifts - ie excluding free days - is crucial here: otherwise the solver will add arbitrarily free days

for c in cycles:
    for w in cycleWeeks:
        for d in Weekdays:
            for k in set_classes: 
                shiftClass = dict_shifts_by_class[k]  ### CLARIFY: used as Boolean?
                # sum_x_for_class is the expression sum(x[c,w,d,sh] for sh in class_shifts)
                # Add equality: class_assigned[c,w,d,k] - sum_x_for_class == 0
                if shiftClass: # run the following code only for non-empty lists of shifts in class k
                    modelCycle.addConstr(
                        class_assigned[c, w, d, k] - gp.quicksum(x[c, w, d, sh] for sh in shiftClass) == 0,
                        name=f"LinkClass_c{c}_w{w}_d{d}_k{k}"
                    )
                else:
                    # if no shifts for this class (shouldn't happen by definition), force 0 (just a safety net)
                    modelCycle.addConstr(class_assigned[c, w, d, k] == 0,
                                         name=f"LinkClassEmpty_c{c}_w{w}_d{d}_k{k}")
                    
# class_diff constraints for consecutive days: counting class changes (looping over cycle>week>day)
# TASK: to be added: loop from last active week to first day of first week
for c in cycles:
    #for w in range(1, MAX_CYCLE_WEEKS-1):   # OLD BUG: stopped two weeks early, so the
    #                                        # transitions of the last two weeks were never
    #                                        # counted and the solver could hide class chaos there
    for w in cycleWeeks:                     # FIX: cover every week
        for d in Weekdays:
            # determine next day index (wrap 7 -> 1)
            if d <= 6:
                d_next = d + 1
                w_next = w
            #else:  # d == 7                          # OLD: no guard, w+1 exploded past the last index
            elif w < MAX_CYCLE_WEEKS:  # d == 7: Sunday -> Monday of the NEXT week
                d_next = 1
                w_next = w+1  # use first day of next cycleWeek
            else:
                continue  # Sunday of the very last week: there is no next week here;
                          # the wrap back to week 1 is handled by the RING CLOSURE cell below
            for k in set_classes:
                # diff >= class_assigned(c,w,d,k) - class_assigned(c,w_next,d_next,k)
                modelCycle.addConstr(
                    #class_diff[c, w, d, k] >= class_assigned[c, w, d, k] - class_assigned[c, w_next, d_next, k],  # OLD: ungated
                    # FIX (gating): '- (1 - active_week[...])' switches the constraint off when the
                    # next day's week is inactive. Without it, stepping from the last active week
                    # into the empty rest of the cycle was counted as a phantom class change.
                    class_diff[c, w, d, k] >= class_assigned[c, w, d, k] - class_assigned[c, w_next, d_next, k] - (1 - active_week[c, w_next]),
                    name=f"ClassDiffPos_c{c}_w{w}_d{d}_k{k}"
                )
                # diff >= class_assigned(c,w_next,d_next,k) - class_assigned(c,w,d,k)
                modelCycle.addConstr(
                    #class_diff[c, w, d, k] >= class_assigned[c, w_next, d_next, k] - class_assigned[c, w, d, k],  # OLD: ungated
                    class_diff[c, w, d, k] >= class_assigned[c, w_next, d_next, k] - class_assigned[c, w, d, k] - (1 - active_week[c, w_next]),
                    name=f"ClassDiffNeg_c{c}_w{w}_d{d}_k{k}"
                )
# Note: diff variables will be 0 when class is same, and 1 when class differs for that k.
# When classes differ (A vs B), two diffs (for A and B) become 1, so sum_k diff = 2.

class_changes_total = 0.5 * gp.quicksum(class_diff[c, w, d, k] for c in cycles for w in cycleWeeks for d in Weekdays for k in set_classes)

In [27]:
### seems to be duplicated cell ???
# # link class_assigned to x
# # for each class k, class_assigned equals the sum of x over all shifts that belong to that class
# # this enforces that class_assigned is 1 exactly when a shift of that class is chosen on that day
# dict_shifts_by_class = {k: [sh for sh, cls in dict_shift_to_class.items() if cls == k] for k in set_classes}

# for c in cycles:
#     for w in cycleWeeks:
#         for d in Weekdays:
#             for k in set_classes:
#                 class_shifts = dict_shifts_by_class[k]  ### CLARIFY: used as Boolean?
#                 # sum_x_for_class is the expression sum(x[c,w,d,sh] for sh in class_shifts)
#                 # Add equality: class_assigned[c,w,d,k] - sum_x_for_class == 0
#                 if class_shifts:
#                     modelCycle.addConstr(
#                         class_assigned[c, w, d, k] - gp.quicksum(x[c, w, d, sh] for sh in class_shifts) == 0,
#                         name=f"LinkClass_c{c}_w{w}_d{d}_k{k}"
#                     )
#                 else:
#                     # if no shifts for this class (shouldn't happen by definition), force 0
#                     modelCycle.addConstr(class_assigned[c, w, d, k] == 0,
#                                          name=f"LinkClassEmpty_c{c}_w{w}_d{d}_k{k}")
                    
# # class_diff constraints for consecutive days: counting class changes (looping over cycle>week>day)
# # TASK: to be added: loop from last active week to first day of first week
# for c in cycles:
#     for w in range(1, MAX_CYCLE_WEEKS-1):
#         for d in Weekdays:
#             # determine next day index (wrap 7 -> 1)
#             if d <= 6:
#                 d_next = d + 1
#                 w_next = w
#             else:  # d == 7
#                 d_next = 1
#                 w_next = w+1  # use first day of next cycleWeek
#             for k in set_classes:
#                 # diff >= class_assigned(c,w,d,k) - class_assigned(c,w_next,d_next,k)
#                 modelCycle.addConstr(
#                     class_diff[c, w, d, k] >= class_assigned[c, w, d, k] - class_assigned[c, w_next, d_next, k],
#                     name=f"ClassDiffPos_c{c}_w{w}_d{d}_k{k}"
#                 )
#                 # diff >= class_assigned(c,w_next,d_next,k) - class_assigned(c,w,d,k)
#                 modelCycle.addConstr(
#                     class_diff[c, w, d, k] >= class_assigned[c, w_next, d_next, k] - class_assigned[c, w, d, k],
#                     name=f"ClassDiffNeg_c{c}_w{w}_d{d}_k{k}"
#                 )
# # Note: diff variables will be 0 when class is same, and 1 when class differs for that k.
# # When classes differ (A vs B), two diffs (for A and B) become 1, so sum_k diff = 2.


#### Ring closure (the snake is a cycle, not a line)
Spec: "the last shift in the snake and the shift at the very first snake position must also observe their required time constraints". After the last active week the worker starts again at week 1, so Sunday of the last active week is followed by Monday of week 1. The three day-by-day rules (c05 min rest, c03 max consecutive working days, Pesch8 class changes) are closed across that seam here.

In [28]:
# ============================== RING CLOSURE ==============================
# WHY THIS EXISTS: all constraints so far treat the cycle as a straight LINE of
# weeks and simply stop at the last one. In reality the rotation wraps around:
# after the last active week the worker continues with week 1. So the pair
# (Sunday of the LAST ACTIVE week) -> (Monday of week 1) is a real consecutive
# pair of days and must obey the same rules as any other day pair.
#
# THE ONE TRICK USED BY EVERY BLOCK BELOW:
# "is week w the LAST ACTIVE week of cycle c?"
# Weeks activate as a prefix without holes (WeekOrder): 1,1,...,1,0,...,0.
# Therefore   active_week[c,w] - active_week[c,w+1]
#   = 1 exactly at the boundary (1 followed by 0)  -> w IS the last active week
#   = 0 everywhere else (1-1 inside the prefix, 0-0 after it).
# For the very last index there is no w+1, so the indicator is active_week[c,MAX]
# itself. This is a plain linear expression built from EXISTING variables, so no
# new binaries are needed. Each ring constraint is written so that it is active
# when the indicator is 1 and trivially satisfied (relaxed) when it is 0.
def last_active_expr(c, w):
    # 0/1-valued linear expression: 1 iff week w is the last active week of cycle c
    if w < MAX_CYCLE_WEEKS:
        return active_week[c, w] - active_week[c, w + 1]
    return active_week[c, w]

# ---- (a) c05 ring: minimum rest across the wrap ---------------------------
# For every incompatible pair (sh1, sh2): if sh1 sits on the last active Sunday,
# sh2 may not sit on Monday of week 1.
#   x1 + x2 <= 2 - last_active(c,w)
# If w is the last active week the bound is 1 (the usual "not both allowed").
# Otherwise the bound is >= 2 and two binaries can never exceed it -> no effect.
for c in cycles:
    for w in cycleWeeks:
        for (sh1, sh2) in incompatible_pairs:
            modelCycle.addConstr(
                x[c, w, 7, sh1] + x[c, 1, 1, sh2] <= 2 - last_active_expr(c, w),
                name=f"RingMinRest_c{c}_w{w}_{sh1}_{sh2}"
            )

# ---- (b) c03 ring: max consecutive working days across the wrap -----------
# A forbidden run of MAX_CONSEC_DAYS+1 working days can also straddle the seam:
# k days at the END of the last active week + (MAX_CONSEC_DAYS+1-k) days at the
# START of week 1. We enumerate every split k = 1..MAX_CONSEC_DAYS.
# When w is NOT the last active week, the right side is inflated by the full
# window length, which makes the constraint impossible to violate there.
for c in cycles:
    for w in cycleWeeks:
        for k in range(1, MAX_CONSEC_DAYS + 1):
            tail = gp.quicksum(x[c, w, d, sh]                     # last k days of week w
                               for d in range(8 - k, 8) for sh in WorkShifts)
            head = gp.quicksum(x[c, 1, d, sh]                     # first (MAX+1-k) days of week 1
                               for d in range(1, MAX_CONSEC_DAYS + 2 - k) for sh in WorkShifts)
            modelCycle.addConstr(
                tail + head <= MAX_CONSEC_DAYS
                             + (MAX_CONSEC_DAYS + 1) * (1 - last_active_expr(c, w)),
                name=f"RingMaxConsec_c{c}_w{w}_k{k}"
            )

# ---- (c) Pesch8 ring: class change across the wrap ------------------------
# One extra transition per cycle: last active Sunday -> Monday of week 1.
# We REUSE the existing class_diff[c, w, 7, k] variables. This cannot clash with
# their normal use (Sunday -> Monday of week w+1), because that constraint is
# gated OFF exactly when week w+1 is inactive, i.e. exactly when w is the last
# active week. The two uses are mutually exclusive by construction.
for c in cycles:
    for w in cycleWeeks:
        for k in set_classes:
            modelCycle.addConstr(
                class_diff[c, w, 7, k] >= class_assigned[c, w, 7, k] - class_assigned[c, 1, 1, k]
                                          - (1 - last_active_expr(c, w)),
                name=f"RingClassDiffPos_c{c}_w{w}_k{k}"
            )
            modelCycle.addConstr(
                class_diff[c, w, 7, k] >= class_assigned[c, 1, 1, k] - class_assigned[c, w, 7, k]
                                          - (1 - last_active_expr(c, w)),
                name=f"RingClassDiffNeg_c{c}_w{w}_k{k}"
            )

#### Pesch3: minimize day-by-day changes of the shift TYPE
Twin of Pesch8: same machinery, different grouping key (shift type = base shiftID instead of shift class).

In [ ]:
# Pesch3 (Dirk's definition): try to keep the SAME shift (by shiftID) on
# consecutive days. Twin of Pesch8 above; the only difference is the grouping key:
#   Pesch8 groups by shift_class -> "stay in the same comfort category"
#   Pesch3 groups by shift TYPE  -> "stay on the exact same shift"
# With the current data (almost one class per type) both behave similarly; they
# diverge once several types share one class (e.g. all day shifts = one class):
# day-week -> day-weekend is then fine for Pesch8 but still a change for Pesch3.
# freeDay carries no type (same WorkShifts filter as Pesch8), so entering/leaving
# a free day costs half a change; a free block costs the same as a single free day.
# Merging Pesch3+Pesch8 into one builder is refactoring work (S8), done with Dirk.

# type = base shift name without the %_% copy suffix, e.g. "[00day000week]"
dict_shift_to_type = {sh: sh.split("%_%", 1)[0] for sh in WorkShifts}
set_types = sorted(set(dict_shift_to_type.values()))
dict_shifts_by_type = {t: [sh for sh, tt in dict_shift_to_type.items() if tt == t] for t in set_types}

# type_assigned[c,w,d,t] = 1 iff the shift worked on (c,w,d) has type t
type_assigned = modelCycle.addVars(cycles, cycleWeeks, Weekdays, set_types, vtype=GRB.BINARY, name="type_assigned")
# type_diff >= |type_assigned(day) - type_assigned(next day)| per type
type_diff = modelCycle.addVars(cycles, cycleWeeks, Weekdays, set_types, vtype=GRB.BINARY, name="type_diff")

# link type_assigned to x (same pattern as LinkClass in Pesch8)
for c in cycles:
    for w in cycleWeeks:
        for d in Weekdays:
            for t in set_types:
                modelCycle.addConstr(
                    type_assigned[c, w, d, t] - gp.quicksum(x[c, w, d, sh] for sh in dict_shifts_by_type[t]) == 0,
                    name=f"LinkType_c{c}_w{w}_d{d}_{t}"
                )

# count type changes between consecutive days; gating and week handling mirror the
# FIXED Pesch8 loop (all weeks covered, inactive next week switches the pair off)
for c in cycles:
    for w in cycleWeeks:
        for d in Weekdays:
            if d <= 6:
                d_next, w_next = d + 1, w
            elif w < MAX_CYCLE_WEEKS:
                d_next, w_next = 1, w + 1
            else:
                continue  # wrap of the very last week is handled by the ring part below
            for t in set_types:
                modelCycle.addConstr(
                    type_diff[c, w, d, t] >= type_assigned[c, w, d, t] - type_assigned[c, w_next, d_next, t]
                                             - (1 - active_week[c, w_next]),
                    name=f"TypeDiffPos_c{c}_w{w}_d{d}_{t}"
                )
                modelCycle.addConstr(
                    type_diff[c, w, d, t] >= type_assigned[c, w_next, d_next, t] - type_assigned[c, w, d, t]
                                             - (1 - active_week[c, w_next]),
                    name=f"TypeDiffNeg_c{c}_w{w}_d{d}_{t}"
                )

# ring: one wrap transition per cycle (last active Sunday -> Monday of week 1),
# same last_active_expr trick and the same variable-reuse argument as Pesch8's ring
for c in cycles:
    for w in cycleWeeks:
        for t in set_types:
            modelCycle.addConstr(
                type_diff[c, w, 7, t] >= type_assigned[c, w, 7, t] - type_assigned[c, 1, 1, t]
                                         - (1 - last_active_expr(c, w)),
                name=f"RingTypeDiffPos_c{c}_w{w}_{t}"
            )
            modelCycle.addConstr(
                type_diff[c, w, 7, t] >= type_assigned[c, 1, 1, t] - type_assigned[c, w, 7, t]
                                         - (1 - last_active_expr(c, w)),
                name=f"RingTypeDiffNeg_c{c}_w{w}_{t}"
            )

# one real change flips two types (1->0 and 0->1), hence the 0.5 factor
type_changes_total = 0.5 * gp.quicksum(type_diff[c, w, d, t]
                                       for c in cycles for w in cycleWeeks for d in Weekdays for t in set_types)

### objective function(s)

In [ ]:
# OPTION 1 - hierarchical objectives
modelCycle.setObjectiveN(gp.quicksum(active_week[c, s] for c in cycles for s in cycleWeeks), index=0, priority=1)#, GRB.MINIMIZE)
modelCycle.setObjectiveN(class_changes_total, index=1, priority=0)


# OPTION 2 - weightes sum of objectives
# set objective function as weighted sum of various objectives as chosen by the user
# modelCycle.setObjective(
#     W_NB_WORKERS  * gp.quicksum(active_week[c, s] for c in cycles for s in cycleWeeks)
#   + W_PESCH1 * gp.quicksum(dev_pos[c, w] + dev_neg[c, w] for c in cycles for w in cycleWeeks)
#   + W_PESCH2   * gp.quicksum(d2_pos[c, k] + d2_neg[c, k] for c in pair_cycles for k in categories)
#   + W_CHANGEofCLASSES * class_changes_total,
#     GRB.MINIMIZE
#     )


Set parameter TimeLimit to value 90
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (linux64 - "Pop!_OS 24.04 LTS")

CPU model: Intel(R) Core(TM) i5-7300HQ CPU @ 2.50GHz, instruction set [SSE2|AVX|AVX2]
Thread count: 4 physical cores, 4 logical processors, using up to 4 threads

Non-default parameters:
TimeLimit  90

Academic license 2810721 - for non-commercial use only - registered to ar___@student.uni-siegen.de
Optimize a model with 106373 rows, 13033 columns and 406711 nonzeros (Min)
Model fingerprint: 0x94712a68
Model has 2068 linear objective coefficients
Variable types: 124 continuous, 12909 integer (12909 binary)
Coefficient statistics:
  Matrix range     [1e+00, 3e+03]
  Objective range  [5e+00, 5e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 3e+03]

Presolve removed 96737 rows and 5832 columns
Presolve time: 0.19s
Presolved: 9636 rows, 7201 columns, 129817 nonzeros
Variable types: 124 continuous, 7077 integer (7077 binary)
Performing another presolve...
P

### solver configuration

In [ ]:

# improvements outstanding:
    # add various weighted objectives => based on user input

# Solver settings (not part of the model, only how it is solved).
# Per-week fairness made this MIP combinatorially hard: a real objective trade-off plus
# heavy cycle/week symmetry, so proving optimality is very slow while a good solution is
# found in seconds. These params make the run practical:
#   TimeLimit = 10  -> stop after 10s and return the best solution found so far
#   MIPFocus  = 1   -> prioritise finding good feasible solutions over proving the bound
#   Symmetry  = 2   -> aggressively detect and discard symmetric (identical) solutions
# The large MIP gap that remains is a weak lower bound, not a bad schedule.

modelCycle.Params.TimeLimit = 180
modelCycle.Params.MIPFocus = 1
modelCycle.Params.Symmetry = 2

#run optimizer
modelCycle.optimize()
weeks_used = sum(active_week[c, s].X for c in cycles for s in cycleWeeks)
total_dev  = sum(dev_pos[c, w].X + dev_neg[c, w].X for c in cycles for w in cycleWeeks)

# Diagnostic for the "in moeglichst vielen Turni" requirement: count how many active
# weeks hit the 40h target exactly (dev = 0). Confirms that L1 minimisation of the
# per-week deviation concentrates the unavoidable error into few weeks and leaves most
# weeks on target, so no explicit "maximise on-target weeks" constraint is needed.
active_cnt   = sum(1 for c in cycles for w in cycleWeeks if active_week[c, w].X > 0.5)
on_target    = sum(1 for c in cycles for w in cycleWeeks
                   if active_week[c, w].X > 0.5 and dev_pos[c, w].X + dev_neg[c, w].X < 1e-6)
print(f"weeks on target (dev=0): {on_target} / {active_cnt} active weeks")
print(f"active weeks: {weeks_used:.0f}, total deviation (h): {total_dev:.1f}, objective: {modelCycle.objVal:.0f}")
#pesch2_imbalance = sum(d2_pos[c, k].X + d2_neg[c, k].X for c in pair_cycles for k in categories)  # OLD: pairwise metric
pesch2_imbalance = sum(d2_pos[c, k].X + d2_neg[c, k].X for c in cycles for k in categories)
print(f"Pesch2 share deviation, sum over cycles (h): {pesch2_imbalance:.1f}")
pesch8_changes = 0.5 * sum(class_diff[c, w, d, k].X for c in cycles for w in cycleWeeks for d in Weekdays for k in set_classes)
print(f"Pesch8 class changes: {pesch8_changes:.0f}")
pesch3_changes = 0.5 * sum(type_diff[c, w, d, t].X for c in cycles for w in cycleWeeks for d in Weekdays for t in set_types)
print(f"Pesch3 type changes: {pesch3_changes:.0f}")


### results

In [30]:
# output (raw version, to be improved for better readability)
#
# FIX LOG (2026-07-07):
# 1. Filename split-brain: the header used to go to "output_cycle_.csv" (note the
#    extra underscore) while all data rows were appended to "output_cycle.csv",
#    which was never truncated. One file held only a header, the other kept
#    collecting stale rows from every previous run. Everything now goes to
#    output_cycle.csv, truncated once per run.
# 2. Python 3.10 compatibility: nested double quotes inside a double-quoted
#    f-string (f"...{shift_info[sh]["start"]}...") are a SyntaxError before
#    Python 3.12. Inner quotes are single now.
# 3. Column semantics: WorkHours = NET work time of the week (breaks excluded).
#    PresenceHours = gross presence time (Anwesenheit) of the week.
#    The class_<k>_presence_h columns split PresenceHours by shift class and sum
#    exactly to it. Note: reserveShift currently shares class 5 with the weekend
#    day shift, so both land in the same column until the team assigns reserve
#    its own class.
if modelCycle.SolCount > 0:   # TimeLimit -> status TIME_LIMIT, not OPTIMAL, so accept any found solution
    output_string = ""
    out_classes = sorted({dict_shift_to_class[sh] for sh in shift_hours})   # classes that carry hours
    with open(FOLDER_OUTPUT / "output_cycle.csv", "w") as file:             # "w" truncates old runs
        file.write("created: " + str(dt.datetime.now()) + ";\n")
        file.write("FINAL CYCLE:\nMon;Tue;Wed;Thu;Fri;Sat;Sun;WorkHours;Deviation;OnTarget;PresenceHours;"
                   + ";".join(f"class_{k}_presence_h" for k in out_classes) + ";\n")
    for c in cycles:
        if active_cycle[c].X > 0.5:
            with open(FOLDER_OUTPUT / "output_cycle.csv", "a") as file:
                file.write(f"CYCLE: {c}:\n")
            for w in cycleWeeks:
                if active_week[c, w].X > 0.5:
                    for d in Weekdays:
                        for sh in Shifts:
                            if x[c, w, d, sh].X > 0.5:
                                output_string = f"{output_string}{sh.split('%_%', 1)[0]}({shift_info[sh]['start']}-{shift_info[sh]['end']});"
                    # per-week net work hours and deviation from the 40h target (from Pesch1 dev vars)
                    signed_dev = dev_pos[c, w].X - dev_neg[c, w].X
                    week_h     = AVG_WEEKLY_HOURS + signed_dev
                    on_target  = "yes" if abs(signed_dev) < 1e-6 else "no"
                    output_string += f"{week_h:.1f};{signed_dev:+.1f};{on_target};"
                    # gross presence of the week: the reference total for the class columns
                    presence_h = sum(shift_hours[sh] * x[c, w, d, sh].X
                                     for d in Weekdays for sh in shift_hours)
                    output_string += f"{presence_h:.1f};"
                    # presence hours split by shift class (columns sum to PresenceHours)
                    for k in out_classes:
                        class_h = sum(shift_hours[sh] * x[c, w, d, sh].X
                                      for d in Weekdays for sh in shift_hours if dict_shift_to_class[sh] == k)
                        output_string += f"{class_h:.1f};"
                    with open(FOLDER_OUTPUT / "output_cycle.csv", "a") as file:
                        file.write(output_string + "\n")
                    output_string = ""